# Exploring Chemical Space
Explore some clustering and visualisation models and compare them for two different datasets:
- A subset of ChEMBL small molecule entries with a molecular weight between 200 and 500. Since the entire set of ChEMBL entries is quite substantial, 10k entries were selected by random sampling (`chembl_200-500_10k`)
- The result of the PubChem search for "antibiotics" (`pubchem_antibiotics`).

Note: The ChEMBL dataset may still take quite some processing time - depending on your PC. You can do a random sampling of e.g. 3k entries in order to reduce the computational effort.

Tasks:
1) Load and inspect the two datasets `ChEMBL_200-500_10k.csv`. Note that they are fundamentally different.
2) Perform basic data cleaning and be mindful of which data to dismiss (if any). Hint: Think about standardising column names (at least for the relevant ones)
3) Make sure that the SMILES strings are valid - implement a function to clean up the SMILES returning (in a new column or Series) the canonical SMILES if the input is valid, and return `None` if the SMILES in the original data is not valid (). Hint: The `Normalizer` in  rdkit might be quite useful (https://www.rdkit.org/docs/cppapi/classRDKit_1_1MolStandardize_1_1Normalizer.html)
4) Calculate Morgan Fingerprints (radius 2, 2048 bit) from the SMILES strings via mol objects. Make sure not to use the outdated version. You can use either the dataframe, numpy arrays or simple lists for the fingerprints.
5) Run different clustering techniques, e.g. snippets provided for Butina and HDBSCAN. You can also try the scikit-learn models kmeans or dbscan.
6) Use the fingerprints to run UMAP and TSNE dimensionality reductions (snippets provided).
7) Plot the data in scatterplots, using the cluster labels as colour map.
8) Adjust some parameters of the clustering models and apply filters if needed (e.g. only visualise clusters of a size larger than 10) to reach some satisfactory result
9) Visualise a representative molecule (e.g. centroids or centers of clusters, or random :) ) of the three largest clusters for both methods using rdkit


Import dependencies and datasets

In [ ]:
# complete imports if needed for your solution
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from rdkit import Chem
from rdkit.Chem.MolStandardize.rdMolStandardize import Normalizer
from rdkit.Chem import rdFingerprintGenerator
from rdkit import DataStructs
from rdkit.ML.Cluster import Butina
from rdkit.Chem import Draw

from sklearn.cluster import KMeans
from sklearn.cluster import DBSCAN
import hdbscan
import umap
from sklearn.manifold import TSNE

In [ ]:
df = pd.read_csv("chembl_200-500_10k.csv")
# df = pd.read_csv("pubchem_antibiotics.csv")

Data cleaning

In [ ]:
# standardise column names
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_").str.replace("[()€$]", "", regex=True)

In [ ]:
# clean up smiles
df["smiles"] = df["smiles"].astype(str)

def standardize_smiles(smiles: str) -> str | None:
    # parse SMILES
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    # apply standard normalization
    normalizer = Normalizer()
    mol = normalizer.normalize(mol)

    # get canonical SMILES
    return Chem.MolToSmiles(mol)


df["smiles_std"] = df["smiles"].apply(standardize_smiles)
df = df.dropna(subset=["smiles_std"]).drop_duplicates(subset="smiles_std")


In [ ]:
df.shape

Calculate Morgan Fingerprints (`GetMorganGenerator`). You can experiment with other fingerprints (`GetRDKitFPGenerator`) as well and see how they impact the clusters.

In [ ]:
def morgan_fp(mols):
    fpg = rdFingerprintGenerator.GetMorganGenerator(fpSize=2048, radius=2)
    fp = fpg.GetFingerprint(mols)
    return fp

mols = [Chem.MolFromSmiles(s) for s in df["smiles"]]
mols = [m for m in mols if m is not None]

fp_list = [morgan_fp(m) for m in mols]
fp_list = [fp for fp in fp_list if fp is not None]

Butina Clustering: Investigate later how different cutoffs for the similarity affect the clusters.

In [ ]:
# Butina clustering requires distance matrix (distance = Tanimoto similarity); 
# fingerprints provided as list in this snippet - adjust as needed!
dists = []
nfps = len(fp_list)

for i in range(1, nfps):
    similarities = DataStructs.BulkTanimotoSimilarity(fp_list[i], fp_list[:i])
    dists.extend([1-x for x in similarities])

# Apply Butina Clustering

# Apply different thresholds later and see how they affect the clustering
cutoff = 0.6  # Tanimoto similarity threshold; e.g. 04 for larger chemical families, 0.7 for tight analogues...

butina_clusters = Butina.ClusterData(
    dists, # similarity based distance matrix
    nfps, # number of fingerprints
    cutoff,
    isDistData=True
)

print("Number of clusters:", len(butina_clusters))

In [ ]:
# filter out small clusters, rare chemoptypes, ...
clusters_filtered = [c for c in butina_clusters if len(c) >= 10]

butina_labels = np.full(nfps, -1)
for cluster_id, cluster in enumerate(clusters_filtered):
    for id in cluster:
        butina_labels[id] = cluster_id

sizes = [len(c) for c in clusters_filtered]

print("clusters:", len(sizes))
print("mean size:", np.mean(sizes))
print("max size:", np.max(sizes))
print("singletons:", sum(s == 1 for s in sizes))

HDBSCAN Clustering: Inspect how `min_cluster_size` and `min_samples` affect the clusters later on. This might depend quite a lot on the dataset (try 15 and 5 for the PubChem data, <10 and <5 for the ChEMBL)

In [ ]:
# use HDBSCAN for clustering
hdbs_clustering = hdbscan.HDBSCAN(
    min_cluster_size=10,
    min_samples=2,
    metric="euclidean"
)

hdbs_labels = hdbs_clustering.fit_predict(fp_list)
# print("Number of DBSCAN clusters:",
#       len(set(hdbs_labels)) - (1 if -1 in hdbs_labels else 0))
# print("Noise points:", list(hdbs_labels).count(-1))

In [ ]:
print("Number of DBSCAN clusters:",
      len(set(hdbs_labels)) - (1 if -1 in hdbs_labels else 0))
print("Noise points:", list(hdbs_labels).count(-1))

Embeddings: TSNE and UMAP

In [ ]:
# convert fingerprints to numpy
X = np.zeros((nfps, 2048), dtype=int)
for i, fp in enumerate(fp_list):
    DataStructs.ConvertToNumpyArray(fp, X[i])

In [ ]:
# Dimensionality reduction by TSNE
umap_model = umap.UMAP(
    n_neighbors=25,
    min_dist=0.2,
    metric="jaccard",
    random_state=42
)

umap_embedding = umap_model.fit_transform(X)

In [ ]:
# Dimensionality reduction by TSNE
tsne_model = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate="auto",
    init="pca",
    random_state=42
)

tsne_embedding = tsne_model.fit_transform(X)

In [ ]:
tsne_embedding

Visualise the two clustering methods in the two embeddings (e.g. as subplot). Make sure to label the axes properly.

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(
    umap_embedding[:,0],
    umap_embedding[:,1],
    c=butina_labels,
    cmap="tab20",
    s=6
)

plt.title("Chemical space clusters")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))

plt.scatter(
    tsne_embedding[:,0],
    tsne_embedding[:,1],
    c=butina_labels,
    cmap="tab20",
    s=6
)

plt.title("Chemical space clusters")
plt.xlabel("TSNE1")
plt.ylabel("TSNE2")

plt.show()

Visualise representative molecules of the three biggest clusters of both methods.

In [ ]:
# Butina clusters have a center. Simply sort clusters by size
clusters_sorted = sorted(clusters_filtered, key=lambda x: len(x), reverse=True)
top3_clusters = clusters_sorted[:3]
centers = [cluster[0] for cluster in top3_clusters]
center_mols = [mols[i] for i in centers]

In [ ]:
from rdkit.Chem import Draw

img = Draw.MolsToGridImage(
    center_mols,
    subImgSize=(200,200)
)

display(img)

In [ ]:
unique, counts = np.unique(hdbs_labels[hdbs_labels >= 0], return_counts=True)

# sort clusters by size
cluster_sizes = sorted(zip(unique, counts), key=lambda x: x[1], reverse=True)
top3_cluster_ids = [cid for cid, size in cluster_sizes[:3]]


In [ ]:
top3_members = {cid: np.where(hdbs_labels == cid)[0] for cid in top3_cluster_ids}
top3_mols = [mols[i] for i in top3_members]


In [ ]:
top3_mols

In [ ]:
img2 = Draw.MolsToGridImage(
    top3_mols,
    subImgSize=(200,200)
)

display(img2)

## Discussion points
1) What are the characteristics of the chemical spaces described in the two dataset? What is the difference?
2) How do density-based clustering techniques compare to models based on similarity in light of the differences in the datasets?
3) What were the best model parameters for the clustering techniques (i.e. that delivered a meaningful result)
4) Comment on the different dimensionality reduction techniques (again in light of the different dataset characteristics)
5) What was the best / most meaningful combination of dimensionality reduction and clustering methods?
6) Comment on some cheminformatics modelling challenges you may have encountered (e.g. runtime, singleton clusters, paramtersensitivity). What could be done to work around a large number of clusters of small size?
7) On the antibiotics dataset, can you identify some known antibiotic classes / motives in your clusters?
